# SED dependent reddening

[Galametz et al. (2017)](https://www.aanda.org/articles/aa/pdf/2017/02/aa29333-16.pdf) investigate the impact of SED dependent reddening. The Milky Way extinction is a function of the source SED. Because we don't know this ahead of time traditionally this was done by 'dereddening' the measurements assuming a flat SED. Here we investigate the impact of SED dependent reddening on the low dust COSMOS field. This impact will be larger on areas with larger reddening in wide surveys.

Equation 2 in [Galametz et al. (2017)](https://www.aanda.org/articles/aa/pdf/2017/02/aa29333-16.pdf) provides a means to compute reddening dependent on the SED. The essential idea is that SEDs with a blue slope across the filter will be more reddened than SEDs with a red slope or flat spectra. In the paper they recommend computing the reddening for a fixed ebv value and assuming a linear relation between the reddening and the $E(B-V)$ value on that position of the sky. 

In this notebook we use LePHARE classes to compute the SED dependent reddening on a library of SEDs and to investigate how it differs from the generic filter based approach.

In [ ]:
import lephare as lp
import numpy as np
from astropy.table import Table
from scipy.interpolate import interp1d
from matplotlib import pylab as plt
import struct
import timeit
import os

%matplotlib inline

## 1 Run prepare to give us access to library of SEDs

As is typical we set the config values and run prepare to create the required model and magnitude libraries. We update some key config values in order to switch on the SED dependent reddening. These are 

   * EXT\_MW\_CURVE
     (CARDELLI[def] or NONE)
     Extinction curve for the Milky Way extinction (used for the energy balance when adding dust emission). Should be in \$LEPHAREDIR/ext if relative.
   * EXT\_ATMOS\_CURVE
     (NONE[def] or e.g. SB\_calzetti.dat)
     Extinction curve for the atmospheric extinction (used for the energy balance when adding dust emission). Should be in \$LEPHAREDIR/ext if relative.
   * APPLY\_MW\_EXTINCTION
     (NONE[DEF], CLASSIC, GALAMETZ)
     Method to apply the Milky Way extinction to the templates. If CLASSIC, the extinction is applied using the E(B-V) value and the extinction curve. If GALAMETZ, the extinction is applied using the E(B-V) value and the extinction curve, but also taking into account the SED dependence of the extinction (see Galametz et al. 2017). In this case, the MW\_REFERENCE\_MODEL keyword must be set to define the reference SED for which the E(B-V) value is defined.
   * MW\_REFERENCE\_MODEL
     (sed/STAR/PICKLES/b5i.sed[DEF])
     Reference SED for which the E(B-V) value is defined when using the GALAMETZ method to apply the Milky Way extinction. Should be in \$LEPHAREDIR if relative.
   * MW\_GLOBAL\_EBV
     (NONE[def] or float)
     Global E(B-V) value to apply to all templates when using the GALAMETZ method to apply the Milky Way extinction. If 0, the E(B-V) value is read from the file defined by MW\_EBV\_FILE.
   * MW\_EBV\_FILE
     (NONE[def] or string)
     Name of the file containing the E(B-V) values to apply to each template when using the GALAMETZ method to apply the Milky Way extinction. The file should have two columns: source ID and E(B-V) value. Should be absolute path.


In [ ]:
config = lp.default_cosmos_config.copy()
bands = "ugrizy"

config.update(
    {
        "CAT_IN": os.path.join(lp.LEPHAREDIR, "examples/COSMOS_no_mw_correction.in"),
        # We look at a reduced filter list to demonstrate with ugrizy data
        "FILTER_LIST": f"cosmos/u_new.pb,hsc/gHSC.pb,hsc/rHSC.pb,hsc/iHSC.pb,hsc/zHSC.pb,hsc/yHSC.pb",
        # We use single values to deal with reduction in FILTER_LIST
        "ERR_SCALE": "0.02",  # Value from page 12 first paragraph in desprez 2023 diff fro phosphours
        "ERR_FACTOR": "1.5",  # Again from paper - diff for phosphorous - 2 lowered from 1.5 to 1. following chisquared dist
        "FILTER_CALIB": "0",
        "GLB_CONTEXT": np.sum(2 ** np.arange(len(bands))),
        "MABS_CONTEXT": np.sum(2 ** np.arange(len(bands))),
        # Reduced z grid for speed in demonstration
        "Z_STEP": ".02,0.,6.",
        # Reduced EM line dispersion for SPEED to demonstrate
        "EM_DISPERSION": "1.",
        ## THESE ARE THE VALUES THAT RELATE TO THE NEW REDDENING MODEL
        "EXT_ATMOS_CURVE": "NONE",
        "APPLY_MW_EXTINCTION": "GALAMETZ",  # "NO[DEF], CLASSIC, GALAMETZ"
        "EXT_MW_CURVE": "LMC_Fitzpatrick.dat",
        "MW_REFERENCE_MODEL": "sed/STAR/PICKLES/b5i.sed",  # The default - a B5 star
        # "MW_GLOBAL_EBV" : "0.016", # This can also be a file with id and ebv value corresponding to each source:
        "MW_EBV_FILE": os.path.join(
            lp.LEPHAREDIR, "examples/EBV_MW.in"
        ),  # FILE with IDs corresponding to the input and EBV values.
    }
)

In [ ]:
# Download the required data
lp.data_retrieval.get_auxiliary_data(
    keymap=config,
    # We need to have the additional files required for the Galametz method
    additional_files=[
        "examples/COSMOS_no_mw_correction.in",
        "examples/EBV_MW.in",
        "ext/LMC_Fitzpatrick.dat",
        "sed/STAR/PICKLES/b5i.sed",
    ],
)

In [ ]:
# Check the config values
config

In [ ]:
# Because we are dealing with some low level functionality we also need the keymap in the native LePHARE format
keymap = lp.all_types_to_keymap(config)

### Make the Model libraries

As for a typical run we run filter, sedtolib, and mag_mal to build the binary files. During those runs we now compute the extinction per band in addition to the band_pass_correction during mag_gal. When you initialize the PhotoZ object the band pass corrections are normalized to the reference model which is typically a B5 star.

In [ ]:
lp.prepare(keymap)

In [ ]:
# This loads the library of SEDs giving us access to the reddening per model
photz = lp.PhotoZ(keymap)

At this stage the EBV values per source have already been set as the EBV file was set at the input time. We can therefore inspect each value per source.

In [ ]:
sources = photz.read_photoz_sources()
photz.read_mw_ebv(sources)
sources[0].mw_ebv
mw_ebv_vals = np.array([s.mw_ebv for s in sources])
plt.hist(mw_ebv_vals, bins=30)
plt.xlabel("$E(B-V)$ [mag]")

## 2. Compute reddening 

We want to compute the baseline reddening as well as the per SED values for comparison. We do this using the original method. This function simply returns a global value for each filter. 

### Start with traditional values

Here we just get one value per band as a baseline "CLASSIC" run. These could be applied to the input catalogue as was done or applied to the models by setting APPLY_MW_EXTINCTION to CLASSIC.

In [ ]:
all_filters, _, _, baseline_albd = lp.classic_extinction_values(config)
baseline_albd

### Look at an example SED

We want to see how they vary over a filter and how the product differs from the pure filter and therefore impacts the reddening.

In [ ]:
# Pick one model to focus on
n_model = 20
# ex_sed=observe(photz, 5, mag)
keymap["t"] = lp.keyword("t", "Q")
qso_mag = lp.GalMag(keymap)
ex_sed = photz.fullLib[n_model]
print(f"Model {n_model} at redshift z={photz.zLib[n_model]} has its z=0 model at index {ex_sed.index_z0}\n")
ex_sed.lamb_flux = photz.fullLib[ex_sed.index_z0].lamb_flux
ex_sed.generate_spectra(photz.zLib[n_model], 1)
x, y = ex_sed.data()
plt.loglog(x, y)

In [ ]:
# The first example is a QSO
ex_sed.is_qso()

We can then inspect the model dependent extinction values for any given model and compare them to the general values 

In [ ]:
# After running the "prepare" function above, every SED in the library has the milky way extinction values in each band
ex_sed.milky_way_extinction

In [ ]:
bands

In [ ]:
fig, ax = plt.subplots()
filter_means = np.array([f.lambdaEff() for f in photz.allFilters])
for n, b in enumerate(bands):
    y = np.array([s.milky_way_extinction[n] - baseline_albd[n] for s in photz.fullLib])
    y = y[np.isfinite(y)]
    if len(y) < 5:
        continue

    ax.violinplot([y], positions=[filter_means[n]], widths=150, bw_method=0.1)

# FIX LIMITS EXPLICITLY

xmin = filter_means.min()
xmax = filter_means.max()
pad = 0.05 * (xmax - xmin)
ax.set_xlim(xmin - pad, xmax + pad)

ax.set_xlabel("Filter mean wavelength [$\\AA$]")
ax.set_ylabel("Difference in reddening")
plt.title("Difference between standard value and SED dependent value for all models")
# TOP AXIS
fig.canvas.draw()
ax_top = ax.twiny()
ax_top.set_xlim(ax.get_xlim())
ax_top.set_xticks(filter_means)
ax_top.set_xticklabels(bands)

## 2.1 Use the python function to get the extinction values

This runs prepare and loads the values per band. These are already computed by lephare.prepare and can also be accessed by interacting directly with the lephare.PhotoZ library of SEDs. We also have a helper function Python side to perform the full calculation and return the results:

In [ ]:
lp.compute_model_reddening?

In [ ]:
import time

start = time.time()
# Get the array of albd values for each model for each band
albd_lib = lp.compute_model_reddening(keymap)
end = time.time()

In [ ]:
print(
    f"Reddening calculation took {end-start:.2f} seconds or {(end-start)/albd_lib.shape[0]*1000:.3f} milliseconds per model"
)

In [ ]:
albd_lib.shape  # models by filters

### Look at distribution of extinction values

Note that the bluer bands have a peak at the low end where the SED has dropped out of the band due to redshifting and the code uses the reddest possible SED in such a case.

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 5]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
for n, b in enumerate("ugrizy"):
    data = albd_lib.T[n][~np.isnan(albd_lib.T[n]) & (albd_lib.T[n] > 0)]
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data) / counts.max(),
        label=f"${b}$",
        color=colors[n],
        linewidth=2,
        alpha=0.5,
    )
    plt.axvline(baseline_albd[n], color=colors[n])
plt.legend(loc="upper left")
plt.ylabel("Relative density", fontsize=14)
plt.xlabel("Reddening $A_{\lambda}/E_{B-V}$ [mag]", fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

In [ ]:
# Get masks to access the object types
qsos = np.array([s.is_qso() for s in photz.fullLib])
gals = np.array([s.is_gal() for s in photz.fullLib])
stars = np.array([s.is_star() for s in photz.fullLib])
ebvs = np.array([s.ebv for s in photz.fullLib])

In [ ]:
np.sum(np.isclose(ebvs, 0.0)), len(ebvs)

In [ ]:
model_numbers = np.array([s.nummod for s in photz.fullLib])

In [ ]:
# This is the b5 star that determines the BPC normalisation
np.sum(stars & (model_numbers == 15))

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 5]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
for n, b in enumerate("ugrizy"):
    data = albd_lib[qsos].T[n][~np.isnan(albd_lib[qsos].T[n]) & (albd_lib[qsos].T[n] > 0)]
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data) / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n], color=colors[n])
plt.legend(loc="upper right")
plt.ylabel("relative density")
plt.xlabel("Reddening $A_{\lambda}/E_{B-V}$ [mag]")
plt.title("QSOs")

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 5]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
for n, b in enumerate("ugrizy"):
    data = albd_lib[gals].T[n][(~np.isnan(albd_lib[gals].T[n])) & (albd_lib[gals].T[n] > 0)]
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data) / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n], color=colors[n], linestyle="--")
plt.legend(loc="upper right")
plt.ylabel("relative density")
plt.xlabel("Reddening $A_{\lambda}/E_{B-V}$ [mag]")
plt.title("Galaxies")

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 5]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
for n, b in enumerate("ugrizy"):
    data = albd_lib[stars].T[n][(~np.isnan(albd_lib[stars].T[n])) & (albd_lib[stars].T[n] > 0)]
    # counts, bins = np.histogram(data, bins=bins)
    plt.hist(data, bins=bins, alpha=0.5, label=b, color=colors[n], linewidth=2)
    plt.axvline(baseline_albd[n], color=colors[n])
plt.legend(loc="upper right")
plt.ylabel("relative density")
plt.xlabel("Reddening $A_{\lambda}/E_{B-V}$ [mag]")
plt.title("Stars")

## 2.2 Band pass correction

We also have a new function for getting the band pass correction based on hard coded B and V bands which are run on every object cpp side. This determines the scaling of the reddening since reddening models are normalised by the $E_{SED}(B-V)$ value of the model with respect to a reference model. There is one value per model which scales the $E(B-V)$ on at the position on the sky.

In [ ]:
lp.compute_band_pass_correction?

In [ ]:
t1 = time.time()
band_pass_correction = lp.compute_band_pass_correction(config)  #
t2 = time.time()

In [ ]:
print(
    f"BPC calculation took {t2-t1:.0f} seconds or {(t2-t1)/band_pass_correction.shape[0]*1000:.3f} milliseconds per model"
)

Note that that function reran prepare

In [ ]:
band_pass_correction

In [ ]:
plt.hist(band_pass_correction, bins=30)
plt.xlabel("bpc$_{SED}$")

In [ ]:
elliptical = gals & (model_numbers < 8)

The following figures correspond to figure 3 in Galametz et al.

In [ ]:
plt.scatter(np.array(photz.zLib)[elliptical], band_pass_correction[elliptical])
plt.plot([0, 2], [1, 1])
plt.xlim([0, 2])
plt.ylim([0.8, 1.2])
plt.xlabel("z")
plt.ylabel("bpc$_{SED}$")
plt.title("Ellipticals")

In [ ]:
plt.scatter(np.array(photz.zLib)[elliptical], band_pass_correction[elliptical], s=0.2)
plt.plot([0, 6], [1, 1])
plt.xlim([0, 6])
plt.ylim([0.2, 2])
plt.xlabel("z")
plt.ylabel("bpc$_{SED}$")
plt.title("Ellipticals")

In [ ]:
albd_lib.shape, band_pass_correction.shape

### Check the redshift dependence of the values

In [ ]:
# We will look at redshifts to 2, 3 and the limit for the run at 6
red_2 = np.array(photz.zLib) < 2
red_3 = np.array(photz.zLib) < 3

We adopt an example E(B-V) of 0.1 to show the impact of the band pass correction as was done in Figure 8 of Galametz et al. (2017)

In [ ]:
galebv_ex = 0.1

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 0.6]
bin_width = 0.005
bins = np.arange(range[0], range[1] + bin_width, bin_width)

m = gals & red_2
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="-.")
plt.legend(loc="upper right")
plt.ylabel("# models per 0.005 mag")
plt.xlabel("Extinction $A_X$ [mag]")
plt.title("Galaxies at z<2 without bpc")

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
range = [0, 0.6]
bin_width = 0.005
bins = np.arange(range[0], range[1] + bin_width, bin_width)
galebv_ex = 0.1
m = gals & red_2
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="-.")
plt.legend(loc="upper right")
plt.ylabel("# models per 0.005 mag")
plt.xlabel("Extinction $A_X$ [mag]")
plt.title("Galaxies at z<2")

Or alternatively use an artificial E(B-V) of 1 to plot the values scaled by the E(B-V) of the dust map

In [ ]:
galebv_ex = 1.0

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
plt.rcParams.update({"font.size": 14})
range = [0, 6]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
galebv_ex = 1.0
m = gals & red_3
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]  # *1.018
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="--")
print("Galaxies to z=3")
plt.legend(loc="upper right", fontsize=8)
plt.ylabel("# models per 0.05 mag")
plt.xlabel("Extinction $A_X/E(B-V)$ [mag]")
plt.title("Galaxy templates $z<3$")
plt.xlim(range)

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
plt.rcParams.update({"font.size": 14})
range = [0, 6]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)

m = qsos & red_3
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]  # *1.018
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="--")
plt.legend(loc="upper right", fontsize=8)
plt.ylabel("# models per 0.05 mag")
plt.xlabel("Extinction $A_{X}/E$(B-V) [mag]")
plt.title("QSO templates $z<3$")
plt.xlim(range)
print("QSO")

In [ ]:
albd_lib

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
plt.rcParams.update({"font.size": 14})
range = [0, 6]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
galebv_ex = 1.0
m = qsos
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]  # *1.018
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="--")
plt.legend(loc="upper right", fontsize=8)
plt.ylabel("# models per 0.05 mag")
plt.xlabel("Extinction $A_{X}/E$(B-V) [mag]")
plt.title("QSO templates $z<6$")
plt.xlim(range)
print("QSO")

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
plt.rcParams.update({"font.size": 14})
range = [0, 6]
bin_width = 0.05
bins = np.arange(range[0], range[1] + bin_width, bin_width)
galebv_ex = 1.0
m = gals
for n, b in enumerate("ugrizy"):
    data = albd_lib[m].T[n]
    data = data * galebv_ex / band_pass_correction[m]  # *1.018
    counts, bins = np.histogram(data, bins=bins)
    plt.hist(
        data,
        bins=bins,
        weights=np.ones_like(data),  # / counts.max(),
        alpha=0.5,
        label=b,
        color=colors[n],
        linewidth=2,
    )
    plt.axvline(baseline_albd[n] * galebv_ex, color=colors[n], linestyle="--")
plt.legend(loc="upper right")
plt.ylabel("# models per 0.05 mag")
plt.xlabel("Extinction $A_{X}/E$(B-V) [mag]")
plt.title("Galaxy templates $z<6$")
plt.xlim(range)

### Look at some individual models and how the BPC changes with redshift

This is equivalent to figure 3 in Galametz et al. (2017)

In [ ]:
fig_size = 5
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
plt.rcParams.update({"font.size": 14})
gal_models = [1, 13]
gal_names = ["Ell", "Sc"]

qso_models = [7, 28]
qso_names = ["Seyfert 1.8", "QSO"]
galebv_ex = 0.1
band_number = 1
# Lets just look at objects with intrinsic ebv =0 for clarity
model_ebvs = np.array([m.ebv for m in photz.fullLib])
range = [0, 6]
for n, g in enumerate(gal_models):
    # print(g)
    mod_mask = np.array([s.nummod == g for s in photz.fullLib])
    mod_mask &= np.array([s.is_gal() for s in photz.fullLib])
    mod_mask &= model_ebvs == 0.0
    mod_reds = np.array(photz.zLib)[mod_mask]
    # data=albd_lib[mod_mask].T[band_number]
    # data=data*galebv_ex*band_pass_correction[mod_mask]
    data = band_pass_correction[mod_mask]
    plt.plot(mod_reds, data, label=gal_names[n])

for n, q in enumerate(qso_models):
    # print(q)
    mod_mask = np.array([s.nummod == q for s in photz.fullLib])
    mod_mask &= np.array([s.is_qso() for s in photz.fullLib])
    mod_mask &= model_ebvs == 0.0
    mod_reds = np.array(photz.zLib)[mod_mask]
    # data=albd_lib[mod_mask].T[band_number]
    # data=data*galebv_ex*band_pass_correction[mod_mask]
    data = band_pass_correction[mod_mask]
    plt.plot(mod_reds, data, label=qso_names[n])

# plt.plot(range,[baseline_albd[band_number],baseline_albd[band_number]],c='k',linestyle='--')
plt.plot(range, [1, 1], c="k", linestyle="--")
plt.xlim(range)
plt.ylim([0.0, 1.2])
plt.xlabel("Redshift")
plt.ylabel("band pass correction")
plt.legend(fontsize=8)

## 3. Compute photoz and compare

We want to look at some actual outputs redshifts to see how it impacts results

In [ ]:
# Most of the other example use "dereddened" fluxes to account for redening using the traditional method.
# Here we need to use the raw observations
input_table = Table.read(os.path.join(lp.LEPHAREDIR, "examples/COSMOS_no_mw_correction.in"), format="ascii")

In [ ]:
cols = [input_table.colnames[0]] + input_table.colnames[3:15] + input_table.colnames[-3:]
input_table = input_table[cols]

In [ ]:
input_table[:5]

In [ ]:
len(input_table)

## Run process with the reddening per model

We do this with the reddened measurements in addition to without and using the reddened models instead

In [ ]:
lp.prepare(config)
t1 = time.time()
out_galametz, _ = lp.process(config, input_table)
t2 = time.time()
# Run again using the classic method for comparison
lp.prepare({**config, "APPLY_MW_EXTINCTION": "CLASSIC"})
t3 = time.time()
out_classic, _ = lp.process({**config, "APPLY_MW_EXTINCTION": "CLASSIC"}, input_table)
t4 = time.time()

In [ ]:
print(f"Time per object with GALAMETZ reddening is a factor  {(t4-t3)/(t2-t1):.2f} larger")

In [ ]:
# How many results changed by more than 0.01?
zspec = out_classic["ZSPEC"]
z1 = out_classic["Z_BEST"]
z2 = out_galametz["Z_BEST"]
np.sum(np.abs(z1 - z2) > 0.01) / len(z1)

In [ ]:
m = np.abs(z1 - z2) > 0.001
plt.plot([0, 2], [0, 2], c="r")
plt.scatter(z1[m], z2[m], s=1)
plt.xlabel("z (Traditional method)")
plt.ylabel("z (Galametz)")
plt.xlim([0, 2])
plt.ylim([0, 2])

### How does the difference compare to general comparison with spectroscopic redshift

At these values of $E(B-V)$ the differences due to the method are smaller than the intrinsic scatter.

In [ ]:
plt.plot([0, 6], [0, 6], c="r")
plt.scatter(zspec[m], z1[m], s=1)
# plt.plot([0,2],[0,2])
plt.ylabel("z (Traditional method)")
plt.xlabel("z (Spec)")
plt.xlim([0, 6])
plt.ylim([0, 6])

In [ ]:
plt.plot([0, 6], [0, 6], c="r")
plt.scatter(zspec[m], z2[m], s=1)
# plt.plot([0,2],[0,2])
plt.ylabel("z (Galametz)")
plt.xlabel("z (Spec)")
plt.xlim([0, 6])
plt.ylim([0, 6])

### How many models changed as a result of the new method?

In [ ]:
same_gal = out_classic["MOD_BEST"] == out_galametz["MOD_BEST"]
same_qso = out_classic["MOD_QSO"] == out_galametz["MOD_QSO"]

In [ ]:
np.sum(~same_gal) / len(same_gal)

In [ ]:
np.sum(~same_qso) / len(same_gal)

In [ ]:
plt.scatter(z1, z2 - z1, s=5)
plt
plt.xlim([0, 6])
plt.ylim([-0.05, 0.05])
plt.plot([0, 6], [0, 0], c="r")
plt.xlabel("Z_BEST (Traditional)")
plt.ylabel("$\Delta$ Z_BEST (Galametz- Traditional)")

plt.title("Galaxies")

### Compare stats for the traditional method and the galaxy sample

In [ ]:
def sigma_nmad(z1, z2):
    maskPos = (z1 > 0.02) & (z2 > 0)  # & (z1 < 1)
    delz = (z2 - z1) / (1 + z1)
    bias = np.nanmedian(delz[maskPos])
    med_delz = np.nanmedian(z2[maskPos] - z1[maskPos])
    delz_unbiased = (z2 - z1 - med_delz) / (1 + z1)
    outlier_frac = np.sum(np.abs(delz[maskPos]) > 0.15) / np.sum(maskPos)
    sigma_nmad_unbiased = 1.48 * np.nanmedian(np.abs(delz_unbiased[maskPos]))

    return {"sigma_nmad_unbiased": sigma_nmad_unbiased, "outlier_frac": outlier_frac, "bias": bias}

In [ ]:
# Classic
sigma_nmad(zspec, z1)

In [ ]:
# Galametz
sigma_nmad(zspec, z2)

Stats for the Galametz method and the galaxy sample

How long does the model reddening take?

In [ ]:
id, flux, flux_err, context, zspec, string_data = lp.table_to_data(config, input_table)
i = 0
one_obj = lp.onesource(i, photz.gridz)
one_obj.readsource(str(id[i]), flux[i], flux_err[i], context[i], zspec[i], str(string_data[i]))
photz.prep_data(one_obj)

In [ ]:
%timeit redened_flux=one_obj.redden_flux(photz.flux,albd_lib)

How does that compare to muliplying using numpy?

In [ ]:
%timeit red_num=photz.flux/10**(albd_lib/2.5)